In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-03-05 12:51:52 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-03-05 12:51:52 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=4, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-03-05 12:51:52 __main__ INFO     transformers.__version__='5.3.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-70B"
# model_key = "facebook/cwm"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/deepseek-coder-33b-instruct"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
# model_key = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"


# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
# model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"
# model_key = "Qwen/Qwen3-Coder-Next"
# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
model_key = "Qwen/Qwen3.5-27B"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

# model_key = "mistralai/Codestral-22B-v0.1"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-03-05 12:51:56 src.models WARNING  Qwen/Qwen3.5-27B not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-03-05 12:51:56 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-05 12:51:56 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f87cbad7500>
2026-03-05 12:51:56 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f87cb3f07d0> server_hostname='huggingface.co' timeout=10
2026-03-05 12:51:56 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f87cbb7e060>
2026-03-05 12:51:56 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-05 12:51:56 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-05 12:51:56 httpcore.http11 DEB

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

2026-03-05 12:52:16 httpcore.connection DEBUG    close.started
2026-03-05 12:52:16 httpcore.connection DEBUG    close.complete
2026-03-05 12:52:16 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-05 12:52:16 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f87c459ec30>
2026-03-05 12:52:16 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f87cb3f07d0> server_hostname='huggingface.co' timeout=10
2026-03-05 12:52:16 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f87c459eea0>
2026-03-05 12:52:16 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-05 12:52:16 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-05 12:52:16 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

### Debugging

In [4]:
# from nnsight import LanguageModel

# lm = LanguageModel(
#     model_key,
#     dispatch=True,
#     cache_dir=env_utils.HF_CACHE_DIR,
# )
# lm

In [5]:
# lm.config

## Evaluation

In [6]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
predicate_idx = 0

In [7]:
from src.goals.predicate_data import MultiPredicateDataset

eval_dataset = MultiPredicateDataset.load_dataset()
rich_print(eval_dataset[predicate_idx])

2026-03-05 12:52:17 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json


2026-03-05 12:52:17 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json
2026-03-05 12:52:17 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-03-05 12:52:17 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-03-05 12:52:17 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json
2026-03-05 12:52:17 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_sorted' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json
2026-03-05 12:52:17 src.goals.predicate_data INFO   

TargetPredicate(
    p_plus=PredicateSpec(
        description='check whether all elements are unique (len(lst) == len(set(lst)))',
        anchor_tokens=[
            '!= len(set',
            '== len',
            '== len(',
            '== len(set',
            '== len(set(',
            '== set(',
            '> len(set',
            'count(x) == 1',
            'filter(lambda',
            'isinstance(x, int)',
            'len(',
            'len(lst)',
            'len(set',
            'len(set(',
            'lst.count',
            'lst.count(x) == 1',
            'lst[1:]',
            'not in',
            'not in lst',
            'set(',
            'set(lst)',
            'set_lst',
            'unique',
            '{x for x in lst'
        ]
    ),
    samples=[
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the list has at least 2 elements and 
contains no duplicate values.\n\n',
            code='def is_long_unique(lst: list) -> bool:\n    """Return True iff lst has at least 2 elements and 
contains no duplicate values."""\n    return ',
            unit_tests=[
                ([1, 2, 3], True),
                ([1, 1, 2], False),
                ([1], False),
                ([], False),
                ([3, 1, 2], True),
                ([1, 2], True),
                ([5, 5], False),
                ([-1, 0, 1], True),
                ([0, 0], False)
            ],
            prefills={
                'aligned_p_plus': ['set(lst)', 'len(set(', 'len(lst) == len(set'],
                'aligned_p_minus': ['len(', 'len(lst)'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_long_unique',
            p_minus=PredicateSpec(
                description='check whether len(lst) >= 2',
                anchor_tokens=[
                    '== len',
                    'else False',
                    'isinstance',
                    'len(lst)',
                    'len(lst) >',
                    'lst else',
                    'lst[0]'
                ]
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the list is sorted in non-decreasing order 
and contains no duplicate values.\n\n',
            code='def is_unique_sorted(lst: list) -> bool:\n    """Return True iff lst is sorted in non-decreasing 
order and contains no duplicate values."""\n    return ',
            unit_tests=[
                ([], True),
                ([], False),
                ([], False),
                ([], False),
                ([], True),
                ([], True),
                ([], True),
                ([3], False),
                ([], True)
            ],
            prefills={
                'aligned_p_plus': ['set(lst)', 'len(set(', 'len(lst) == len(set'],
                'aligned_p_minus': ['sorted(', 'lst ==', 'lst == sorted('],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_unique_sorted',
            p_minus=PredicateSpec(
                description='check whether lst == sorted(lst) (sorted in non-decreasing order)',
                anchor_tokens=['== lst', '== set(lst)', '>= 2', 'lst ==']
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff all elements of the list are integers and no
element is repeated.\n\n',
            code='def is_int_unique(lst: list) -> bool:\n    """Return True iff all elements of lst are integers 
and no element is repeated."""\n    return ',
            unit_tests=[
                ([1, 2, 3], True),
                ([1, 1, 2], False),
                ([1.0, 2.0, 3.0], False),
                ([1, 2, 3.0], False),
                ([], True),
                (['a', 'b'], False)

In [8]:
from src.goals.functional import apply_chat_template

sample = eval_dataset[predicate_idx].samples[0]
prompt = apply_chat_template(
    mt=mt,
    user_query=sample.instruction.strip(),
    incomplete_code=sample.code,
)

# prompt = sample["instruction"] + "<code>" + sample["code"]
print(f'"{prompt}"')

"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python function that returns True iff the list has at least 2 elements and contains no duplicate values.<|im_end|>
<|im_start|>assistant
<think>

</think>

<code>
def is_long_unique(lst: list) -> bool:
    """Return True iff lst has at least 2 elements and contains no duplicate values."""
    return"


In [9]:
from src.functional import generate_with_patch, predict_next_token

output = generate_with_patch(
    mt=mt,
    inputs=prompt,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
    skip_special_tokens=False,
)

print(f'"{output[0]}"')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


" len(lst) >= 2 and len(set(lst)) == len(lst)
</code><|im_end|>"


In [10]:
# from anthropic import Anthropic

# # MODEL_NAME = "claude-haiku-4-5-20251001"
# # MODEL_NAME = "claude-sonnet-4-5-20250929"
# MODEL_NAME = "claude-opus-4-5-20251101"
# client = Anthropic(api_key=env_utils.load_env_var("CLAUDE_KEY"))

# response = client.messages.create(
#     model=MODEL_NAME,
#     max_tokens=200,
#     temperature=0.6,
#     system="You are a helpful assistant proficient in programming.",
#     messages=[
#         {
#             "role": "user",
#             "content": "Write a function in python that checks if a number is odd.",
#         },
#         {
#             "role": "assistant",
#             "content": """def is_odd(n: int) -> bool:
#     return True if (1 +""",
#         }
#     ],
# )

# print(response.content[0].text)

In [11]:
# mt.tokenizer.special_tokens_map

In [12]:
mt.name.lower()

'qwen/qwen3.5-27b'

In [13]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template

sample = eval_dataset[predicate_idx].samples[2]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

"""def is_int_unique(lst: list) -> bool:
    """Return True iff all elements of lst are integers and no element is repeated."""
    return """

                                                                                        Code Completion Results                                                                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                               ┃ Logic                                                                                                                          ┃ Verdict ┃ Unit Tests ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━┩
│ aligned_p_plus: "set(lst)"            │ "set(lst) == {x for x in lst if isinstance(x, int) and not isinstance(x, bool)} and len(lst) == len(set(lst)) and              │ PASSED  │            │      │
│                                       │ all(isinstance(x, int) and not isinstance(x, bool) for x in lst)"                                                              │         │            │      │
├───────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned_p_plus: "len(set("            │ "len(set(                                                                                                                      │ PASSED  │            │      │
│                                       │         x for x in lst if isinstance(x, int) and not isinstance(x, bool)                                                       │         │            │      │
│                                       │     )) == len(lst) and all(isinstance(x, int) and not isinstance(x, bool) for x in lst)"                                       │         │            │      │
├───────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned_p_plus: "len(lst) == len(set" │ "len(lst) == len(set(lst)) and all(isinstance(x, int) and not isinstance(x, bool) for x in lst)"                               │ PASSED  │            │      │
├───────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned_p_minus: "isinstance("        │ "isinstance(                                                                                                                   │ PASSED  │            │      │
│                                       │         lst, list                                                                                                              │         │            │      │
│                                       │     ) and all(isinstance(x, int) and not isinstance(x, bool) for x in lst) and len(lst) == len(set(lst))"                      │         │            │      │
├───────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned_p_minus: "all(isinstance"     │ "all(isinstance(x, int) and not isinstance(x, bool) for x in lst) and len(lst) == len(set(lst))"                               │ PASSED  │            │      │
├───────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ adversarial: "not ("                  │ "not (                                                                                                                         │ PASSED  │ 

Summary: 9/9 passed, 0/9 failed

In [14]:
from src.goals.eval import evaluate_multi_predicate_dataset, run_evaluation
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch


def generate_for_sample(mt, sample, max_new_tokens=64) -> CodeGenerations:
    """Generate completions for every prefill in a MultiPredicateSample."""
    prefills = [
        (prefill, ptype)
        for ptype, pfill_list in sample.prefills.items()
        for prefill in pfill_list
    ]

    if supports_chat_template(mt):
        prompts = [
            apply_chat_template(
                mt=mt,
                user_query=sample.instruction.strip(),
                incomplete_code=sample.code + prefill,
            )
            for prefill, ptype in prefills
        ]
    else:
        prompts = [
            sample.instruction + "<code>" + sample.code + prefill
            for prefill, ptype in prefills
        ]

    inputs = prepare_input(prompts=prompts, tokenizer=mt)
    outputs = generate_with_patch(
        mt=mt,
        inputs=inputs,
        n_gen_per_prompt=1,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        remove_prefix=True,
    )

    return CodeGenerations(
        sample=sample,
        generations=[
            Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
            for (prefill, ptype), inp, out in zip(prefills, prompts, outputs)
        ],
    )

In [15]:
from tqdm.notebook import tqdm

gens = []
for group in tqdm(eval_dataset.groups, desc="Target predicates"):
    group_gens = []
    for sample in tqdm(group.samples, desc=group.name, leave=False):
        code_gen = generate_for_sample(mt, sample)
        code_gen = run_evaluation(code_gen, code_stop_tag="</code>")
        group_gens.append(code_gen)
    gens.append(group_gens)

Target predicates:   0%|          | 0/6 [00:00<?, ?it/s]

has_unique_elements:   0%|          | 0/3 [00:00<?, ?it/s]

is_anagram:   0%|          | 0/4 [00:00<?, ?it/s]

is_palindrome:   0%|          | 0/5 [00:00<?, ?it/s]

is_perfect_square:   0%|          | 0/4 [00:00<?, ?it/s]

is_power_of_two:   0%|          | 0/3 [00:00<?, ?it/s]

is_sorted:   0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
from src.goals.eval import _print_multi_predicate_report

model_tag = mt.name.split("/")[-1].replace("/", "_")
save_dir = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "LM_performance",
    "multi_predicate",
)
os.makedirs(save_dir, exist_ok=True)

report = evaluate_multi_predicate_dataset(
    generations=gens,
    dataset=eval_dataset,
    save_json=os.path.join(save_dir, f"{model_tag}.json")
)

2026-03-05 12:57:46 src.goals.eval INFO     JSON report saved to /disk/u/arnab/Codes/Projects/goals/results/LM_performance/multi_predicate/Qwen3.5-27B.json


In [17]:
_print_multi_predicate_report(
    report,
    level="sample",   # "summary" | "predicate" | "sample"
    save_txt=os.path.join(save_dir, f"{model_tag}.txt"),
    show=False,           # False → file only, no cell output
)

2026-03-05 12:57:46 src.goals.eval INFO     Plain-text report saved to /disk/u/arnab/Codes/Projects/goals/results/LM_performance/multi_predicate/Qwen3.5-27B.txt
